# Syft CLI scan vs library mode: when to use each, and how results differ for multi-arch images

## Purpose

Syft can inventory software in two different ways: run as a standalone command that scans a target and writes out a Software Bill of Materials (SBOM), or embedded as a library inside a larger program that needs package data at runtime. This notebook compares the two modes and shows, with concrete data, why scanning a multi-architecture image once without selecting an architecture gives a misleading package list.

## When to use each

- **CLI scan mode** — use when the SBOM is a build artifact: generate it in CI, store it next to the image, hand it to a scanner or an auditor. The scan is a discrete step with a file as its output.
- **Library mode** — use when package data must drive program logic: a custom gate that compares two images, a service that enriches inventory with internal metadata, or a controller that reacts to new packages. The calling program decides which sources to open, how to combine them, and what to do with each package record.
- **Rule of thumb:** if a human or a downstream tool consumes a document, scan from the command line. If code consumes package records, embed the cataloging step in that code.

## Prerequisites

- Python 3 with the standard library only (`json`, `collections`). No extra packages.
- The comparison below runs on embedded sample payloads, so it executes anywhere. The payloads mirror the shape of real SBOM package lists: name, version, and the architecture each entry was observed on.

In [ ]:
# last_verified: 2026-09-17 · syft n/a
"""Sample per-architecture package lists for one multi-arch image.

Each entry mirrors one SBOM package record: name, version, and the
architecture it was cataloged from. The two lists intentionally share
some packages (same name, sometimes different versions) while each
carries architecture-specific entries the other lacks.
"""
amd64_packages = [
    {"name": "openssl", "version": "3.1.4", "arch": "amd64"},
    {"name": "zlib", "version": "1.2.13", "arch": "amd64"},
    {"name": "libamd-helper", "version": "2.0.1", "arch": "amd64"},
    {"name": "curl", "version": "8.4.0", "arch": "amd64"},
]

arm64_packages = [
    {"name": "openssl", "version": "3.1.4", "arch": "arm64"},
    {"name": "zlib", "version": "1.3.0", "arch": "arm64"},
    {"name": "libarm-helper", "version": "1.9.2", "arch": "arm64"},
    {"name": "curl", "version": "8.4.0", "arch": "arm64"},
]


def index_by_name(packages):
    """Map package name -> set of (version, arch) observed."""
    index = {}
    for pkg in packages:
        index.setdefault(pkg["name"], set()).add((pkg["version"], pkg["arch"]))
    return index


def compare_arch_lists(first, second):
    """Split two per-arch package lists into shared, unique, and drifted names."""
    a, b = index_by_name(first), index_by_name(second)
    names_a, names_b = set(a), set(b)
    shared_identical = sorted(
        n for n in names_a & names_b if a[n] == b[n] or {v for v, _ in a[n]} == {v for v, _ in b[n]} and len(a[n]) == len(b[n]) == 1
    )
    version_drift = sorted(n for n in names_a & names_b if n not in shared_identical)
    return {
        "only_in_first": sorted(names_a - names_b),
        "only_in_second": sorted(names_b - names_a),
        "shared_same_version": shared_identical,
        "shared_version_drift": version_drift,
    }

In [ ]:
result = compare_arch_lists(amd64_packages, arm64_packages)
for key, names in result.items():
    print(f"{key:22s}: {names}")

# What a single unscoped scan hides: union the two lists the way a naive
# merge would, then count how many names carry more than one version.
merged = {}
for pkg in amd64_packages + arm64_packages:
    merged.setdefault(pkg["name"], set()).add(pkg["version"])
multi_version = sorted(n for n, versions in merged.items() if len(versions) > 1)
print("names with arch-dependent versions:", multi_version)

In [ ]:
# Verify: the comparison must surface the known differences.
assert result["only_in_first"] == ["libamd-helper"]
assert result["only_in_second"] == ["libarm-helper"]
assert "zlib" in result["shared_version_drift"]
assert "openssl" in result["shared_same_version"]
assert multi_version == ["zlib"]
print("verify: per-arch comparison behaves as documented")

## What the comparison shows

- **Each architecture has unique packages.** The helper library present on one architecture is absent on the other. A scan scoped to a single architecture never reports the other side's packages, so gating on one scan silently ignores half the image.
- **Shared names can carry different versions.** The compression library ships at different versions per architecture. A merged list that keys on name alone either drops one version or double-counts, and either choice corrupts downstream vulnerability matching.
- **Mode guidance follows from this.** In CLI scan mode, scope each invocation to one architecture and keep one SBOM file per architecture. In library mode, open each architecture as a separate source inside the program and tag every package record with the source it came from before combining anything.

## Common errors

- **Scanning the multi-arch reference once and treating the result as complete.** The scan resolves to whatever the default selection is; the unselected architectures contribute nothing to the SBOM.
- **Merging per-architecture SBOMs on package name.** Shared names with diverging versions collapse into a single row, hiding the version that actually ships on one architecture.
- **Embedding cataloging in code but reusing one accumulator across sources.** Without tagging each record with its source architecture, the program cannot tell a genuine duplicate from an architecture-specific difference.

## References

- [SBOM formats comparison](../docs/sbom-formats-comparison.md)
- [SBOM output formats reference](../docs/sbom-output-formats-reference.md)